# 🔬 Notebook 3: Distributed Lock Manager — Deep Dive: Toy lock manager with fencing

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive — build a toy lock manager with fencing

In [ ]:
import time, threading
from dataclasses import dataclass, field

@dataclass
class LockEntry:
    owner: str
    token: int
    expires_at: float

class LockManager:
    def __init__(self):
        self._locks: dict[str, LockEntry] = {}
        self._next_token = 0
        self._mu = threading.Lock()

    def _now(self): return time.time()

    def acquire(self, name: str, owner: str, ttl_s: float):
        with self._mu:
            existing = self._locks.get(name)
            if existing and existing.expires_at > self._now() and existing.owner != owner:
                return None  # held by someone else
            self._next_token += 1
            entry = LockEntry(owner=owner, token=self._next_token,
                              expires_at=self._now() + ttl_s)
            self._locks[name] = entry
            return {"token": entry.token, "expires_at": entry.expires_at}

    def renew(self, name: str, owner: str, token: int, ttl_s: float):
        with self._mu:
            e = self._locks.get(name)
            if not e or e.owner != owner or e.token != token:
                return None
            if e.expires_at < self._now():
                return None
            e.expires_at = self._now() + ttl_s
            return {"expires_at": e.expires_at}

    def release(self, name: str, owner: str, token: int):
        with self._mu:
            e = self._locks.get(name)
            if e and e.owner == owner and e.token == token:
                del self._locks[name]; return True
            return False

lm = LockManager()
a1 = lm.acquire("nightly-job", "worker-A", ttl_s=1.0)
print("A acquired:", a1)

# B is denied while A holds
print("B denied?:", lm.acquire("nightly-job", "worker-B", ttl_s=1.0))

# Wait for A's lease to expire without renewal
time.sleep(1.1)
a2 = lm.acquire("nightly-job", "worker-B", ttl_s=1.0)
print("B acquired after A expired:", a2, "(token strictly greater)")


In [ ]:
# A "protected resource" that rejects stale-token writes.
class FencedResource:
    def __init__(self):
        self.last_token = 0
        self.state = []
    def write(self, token: int, data: str):
        if token < self.last_token:
            print(f"  REJECT: token {token} < last_seen {self.last_token}")
            return False
        self.last_token = token
        self.state.append((token, data))
        print(f"  ACCEPT: token {token} wrote {data!r}")
        return True

res = FencedResource()
# Simulate the GC pause scenario from notebook 1
res.write(token=1, data="A: initial")
# A pauses. B acquires newer token.
res.write(token=2, data="B: takeover")
# A wakes up and tries to write — MUST be rejected
res.write(token=1, data="A: stale write after pause")
print("Final state:", res.state)


## Deep dive — Redis-only approach (for reference)

```python
# Acquire (atomic)
SET lock:nightly-job worker-A NX PX 30000

# Release (must check owner — use a Lua script for atomicity)
EVAL "if redis.call('get', KEYS[1]) == ARGV[1] \
      then return redis.call('del', KEYS[1]) \
      else return 0 end"  1  lock:nightly-job  worker-A
```

For a **fencing token**, combine with `INCR locks:nightly-job:token` atomically in the same
script. Return that token to the client.

### When Redis is not enough
- Redis master failure right after `SET NX` → failover promotes a replica that *doesn't* have
  the key → two clients can hold the lock. Redlock tries to mitigate; consensus systems don't have this issue.
